# Apache Spark 调优 & 踩坑

> **适用场景**: 生产环境 Spark 作业性能优化、OOM 排查
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录
1. Shuffle Partition 数量调整
2. spark.sql.adaptive.* 配置 (AQE)
3. OOM 排查：Executor vs Driver
4. Speculative Execution
5. 小文件合并策略
6. RDD vs DataFrame vs Dataset 对比
7. 练习题

---
## 1. Shuffle Partition 数量调整

### 为什么重要？
`spark.sql.shuffle.partitions`（默认 **200**）控制 shuffle 后的分区数，直接影响：
- 并行度
- 每个 Task 处理的数据量
- 小文件数量

### 经验公式
```
目标分区数 = max(
    总 Executor Cores × 2~4,          # 保证 CPU 利用率
    总 Shuffle 数据量(GB) × 128~200    # 每个分区约 128MB-200MB
)
```

### 调整方式
```python
# 静态设置（整个 Session 生效）
spark.conf.set('spark.sql.shuffle.partitions', 400)

# 动态：开启 AQE 后自动调整（推荐）
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
```

### 常见问题
| 问题 | 原因 | 解决 |
|------|------|------|
| 分区太少 | 每个 Task 数据过多，OOM | 增大 partitions |
| 分区太多 | Task 调度开销大，产生大量小文件 | 减小 partitions 或开启 AQE |
| 数据倾斜 | 少数分区特别大 | Salting / AQE skew join |

---
## 2. spark.sql.adaptive.* 配置 (AQE)

### AQE 三大功能（Spark 3.0+）

**① 动态合并 Shuffle 分区**
```python
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.minPartitionSize', '1MB')
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '128MB')
# 将过小的 shuffle 分区自动合并，减少小文件
```

**② 动态切换 Join 策略**
```python
spark.conf.set('spark.sql.adaptive.localShuffleReader.enabled', 'true')
# 运行时发现小表，自动切换为 Broadcast Join，跳过 Shuffle
```

**③ 动态处理数据倾斜**
```python
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')
# 倾斜分区自动拆分，与另一侧对应分区做多次 Join
```

### 关键参数速查
```python
spark.conf.set('spark.sql.adaptive.enabled', 'true')                          # 开关
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')       # 合并小分区
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')                 # 倾斜处理
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '134217728') # 128MB
```

---
## 3. OOM 排查：Executor vs Driver

### 先判断是哪里 OOM

**Executor OOM** 特征：
- 错误在 Task 日志：`java.lang.OutOfMemoryError: Java heap space`
- 某些 Stage 失败并重试

**Driver OOM** 特征：
- 整个 Application 崩溃
- 错误在 Driver 日志
- 常发生在 `collect()`, `toPandas()`, 大 broadcast 变量

### Executor OOM 排查 & 修复
```python
# 1. 增大 Executor 内存
spark.conf.set('spark.executor.memory', '8g')        # JVM 堆
spark.conf.set('spark.executor.memoryOverhead', '2g') # 堆外（Python UDF/Arrow）

# 2. 减少每个 Executor 的并发 Task
spark.conf.set('spark.executor.cores', '2')  # 降低并发，每个 Task 有更多内存

# 3. 增大 shuffle 分区（减少每个 Task 的数据量）
spark.conf.set('spark.sql.shuffle.partitions', '800')

# 4. 检查数据倾斜（倾斜分区单个 Task 数据量过大）
# 查看 Spark UI → Stages → Tasks → Input/Shuffle Read Size
```

### Driver OOM 排查 & 修复
```python
# 1. 增大 Driver 内存
spark.conf.set('spark.driver.memory', '4g')

# 2. 避免将大数据 collect 到 Driver
# ❌ df.collect()        # 全量拉到 Driver
# ✅ df.limit(100).collect()  # 采样

# 3. 控制 Broadcast 变量大小
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50MB')  # 默认 10MB

# 4. Python UDF 内存溢出：改用 Pandas UDF（Arrow，减少序列化开销）
```

---
## 4. Speculative Execution

### 什么是 Speculative Execution？
当某个 Task 运行时间显著超过同 Stage 其他 Task 的中位数时，Spark 会在另一个 Executor 上启动**相同 Task 的副本**，先完成者使用，另一个被杀掉。

### 配置
```python
spark.conf.set('spark.speculation', 'true')                    # 开启
spark.conf.set('spark.speculation.multiplier', '1.5')         # 超过中位数 1.5倍触发
spark.conf.set('spark.speculation.quantile', '0.75')          # 75% Task 完成后才检查
spark.conf.set('spark.speculation.minTaskRuntime', '100s')    # Task 至少运行这么久才推测
```

### 注意事项
- **写入幂等性要求**：两个副本都可能写出数据，需保证写操作幂等（如覆盖式写入）
- **数据倾斜 ≠ 慢 Task**：真正倾斜的 Task 需要更多时间处理更多数据，推测执行无法解决根本问题
- 适合：硬件故障导致的偶发慢节点，不适合：数据倾斜

---
## 5. 小文件合并策略

### 小文件问题的根源
- 过多的 shuffle partitions
- 流式写入每个 micro-batch 写一批小文件
- 动态分区写入（每个分区值一个文件）

### 方案一：写入前 repartition/coalesce
```python
# coalesce: 只减少分区，无 shuffle（窄操作）
df.coalesce(10).write.parquet('/output/')

# repartition: 完全重新分区，有 shuffle（保证均匀）
df.repartition(10).write.parquet('/output/')

# 按分区键 repartition，避免动态分区产生过多文件
df.repartition('date', 'region').write.partitionBy('date', 'region').parquet('/output/')
```

### 方案二：AQE 自动合并
```python
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
# 自动将小 shuffle 分区合并，减少输出文件数
```

### 方案三：Delta Lake OPTIMIZE
```sql
OPTIMIZE delta.`/path/to/table`;          -- 合并小文件
OPTIMIZE delta.`/path/to/table` ZORDER BY (user_id);  -- 同时 Z-Order
```

### 方案四：定期 Compaction Job
```python
# 读入并重写，达到目标文件大小（如每个文件 ~128MB）
spark.read.parquet('/path/').repartition(50).write.mode('overwrite').parquet('/path/')
```

---
## 6. RDD vs DataFrame vs Dataset 对比

| 维度 | RDD | DataFrame | Dataset |
|------|-----|-----------|----------|
| 类型安全 | ✅ 编译期 | ❌ 运行时 | ✅ 编译期 |
| Catalyst 优化 | ❌ | ✅ | ✅ |
| Tungsten 内存 | ❌ | ✅ | ✅ |
| 适用语言 | Scala/Java/Python | 所有 | Scala/Java |
| 序列化 | Java/Kryo | Tungsten Row | Encoder |
| 何时使用 | 非结构化数据、细粒度控制 | 结构化数据分析（推荐） | Scala 类型安全场景 |

### 实际建议
- **PySpark**: 始终用 DataFrame/SQL，RDD 几乎没有使用场景
- **Scala**: 推荐 Dataset，兼顾类型安全和性能
- **避免**: 在 DataFrame 上用 `.rdd.map()` 会失去所有优化

---
## 7. 练习题

### Q1 [高频] Spark 作业很慢，你从哪些维度排查？

<details><summary>参考答案</summary>

1. **Spark UI** → Jobs/Stages/Tasks 找最慢的 Stage
2. **数据倾斜**：看 Tasks tab，若某个 Task 远比其他慢且数据量大 → Skew Join / Salting
3. **Shuffle 量**：是否过多 Shuffle？考虑 Broadcast Join
4. **Partition 数**：太少（并行不足）或太多（调度开销+小文件）
5. **GC 时间**：Executor 日志中 GC 时间占比高 → 增加内存或减少对象创建
6. **Spill**：Shuffle spill 到磁盘 → 增加 `spark.executor.memory`
7. **慢节点**：考虑开启 Speculative Execution
</details>

---

### Q2 [高频] 什么情况下开启 AQE？有什么限制？

<details><summary>参考答案</summary>

**开启场景**：
- 数据量不均匀，静态 partition 数难以确定
- 有数据倾斜的 Join
- 输出文件数需要控制

**限制**：
- 需要 Spark 3.0+
- Streaming 不支持 AQE
- 某些复杂 SQL 模式下 skew join 检测可能不准
- 会增加 Planning 时间（需等统计信息收集）
</details>

---

### Q3 [高频] Executor OOM 如何排查和解决？

<details><summary>参考答案</summary>

排查步骤：
1. Spark UI → Stages → 找失败的 Task，看错误日志确认是 OOM
2. 检查 Task 的 Input Size / Shuffle Read Size，判断是否数据倾斜
3. 检查是否有 Python UDF（会有额外的堆外内存消耗）

解决：
- 增加 `spark.executor.memory` 和 `spark.executor.memoryOverhead`
- 减少 `spark.executor.cores`（每个 Task 分到更多内存）
- 增大 `spark.sql.shuffle.partitions`（减少每 Task 数据量）
- 处理数据倾斜（Salting 或开启 AQE skewJoin）
- Python UDF 改为 Pandas UDF（Arrow 序列化，内存更高效）
</details>

---

### Q4 coalesce 和 repartition 的区别，各自适用场景？

<details><summary>参考答案</summary>

- **coalesce(n)**：窄依赖，合并分区，无 Shuffle。只能减少分区数。数据可能不均匀（某些 Task 数据更多）。适合：写入前减少文件数，数据已经相对均匀。
- **repartition(n)**：宽依赖，完全重新 Shuffle 分区，数据均匀。可以增加或减少分区。适合：需要均匀分布数据后再处理。

原则：优先 coalesce（避免 Shuffle），数据倾斜时用 repartition。
</details>

---

### Q5 为什么 Python UDF 慢？如何优化？

<details><summary>参考答案</summary>

Python UDF 慢的原因：
1. **序列化开销**：JVM → Python 进程逐行传递数据（Pickle 序列化）
2. **进程切换**：每条记录都要跨进程传递
3. **无法享受 Catalyst/Tungsten 优化**

优化方案（按推荐度排序）：
1. ✅ **改用内置函数** `pyspark.sql.functions`（在 JVM 内执行，最快）
2. ✅ **Pandas UDF** (`@pandas_udf`)：批量传递 Arrow 格式，减少序列化
3. ✅ **Spark SQL**：纯 SQL 实现逻辑
4. ❌ 避免逐行 Python UDF
</details>